# Support Vector Regression (SVR)

This notebook demonstrates Support Vector Regression using Scikit-learn's Diabetes dataset. It covers baseline SVR, Linear and Polynomial kernels, GridSearchCV, and LinearSVR. Feature scaling is included because SVM models are sensitive to feature magnitudes.

In [ ]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.svm import SVR, LinearSVR
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 1. Load the dataset

In [ ]:
df = datasets.load_diabetes(as_frame=True).frame
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("Missing values by column:\n", df.isnull().sum())

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 2. Scale target and input features

The target is standardized for this learning experiment. Both scalers are fitted only on training data. During cross-validation, the SVR pipeline below fits feature scaling separately within each fold.

In [ ]:
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1)).ravel()
y_test_scaled = y_scaler.transform(y_test.to_numpy().reshape(-1, 1)).ravel()

X_scaler = StandardScaler()
X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

## 3. Baseline SVR

In [ ]:
model = SVR()
model.fit(X_train_scaled, y_train_scaled)

y_train_pred_scaled = model.predict(X_train_scaled)
y_test_pred_scaled = model.predict(X_test_scaled)

print("Training R²:", r2_score(y_train_scaled, y_train_pred_scaled))
print("Test R²:", r2_score(y_test_scaled, y_test_pred_scaled))

## 4. Compare kernels

In [ ]:
# Linear kernel
linear_svr_model = SVR(kernel="linear")
linear_svr_model.fit(X_train_scaled, y_train_scaled)
linear_train_pred_scaled = linear_svr_model.predict(X_train_scaled)
linear_test_pred_scaled = linear_svr_model.predict(X_test_scaled)

print("Linear SVR training R²:", r2_score(y_train_scaled, linear_train_pred_scaled))
print("Linear SVR test R²:", r2_score(y_test_scaled, linear_test_pred_scaled))

In [ ]:
# Polynomial kernel
poly_svr_model = SVR(kernel="poly")
poly_svr_model.fit(X_train_scaled, y_train_scaled)
poly_train_pred_scaled = poly_svr_model.predict(X_train_scaled)
poly_test_pred_scaled = poly_svr_model.predict(X_test_scaled)

print("Polynomial SVR training R²:", r2_score(y_train_scaled, poly_train_pred_scaled))
print("Polynomial SVR test R²:", r2_score(y_test_scaled, poly_test_pred_scaled))

## 5. Hyperparameter tuning using GridSearchCV

The search uses a Pipeline, so scaling is fitted inside each training fold. The held-out test set is reserved for the final evaluation.

In [ ]:
param_grid = {
    "svr__C": [1, 2, 5, 10, 50, 100],
    "svr__kernel": ["rbf", "linear"],
    "svr__epsilon": [0.01, 0.1, 0.2, 0.3, 0.5]
}

svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR())
])

grid_search = GridSearchCV(
    svr_pipeline,
    param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1
)
grid_search.fit(X_train, y_train_scaled)

In [ ]:
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation R²:", grid_search.best_score_)

In [ ]:
best_model = grid_search.best_estimator_
best_y_train_pred_scaled = best_model.predict(X_train)
best_y_test_pred_scaled = best_model.predict(X_test)

print("Selected SVR training R²:", r2_score(y_train_scaled, best_y_train_pred_scaled))
print("Selected SVR test R²:", r2_score(y_test_scaled, best_y_test_pred_scaled))

## 6. Compare with LinearSVR

`LinearSVR` is a separate implementation intended for linear support vector regression. Here, its feature scaling is also placed in a pipeline.

In [ ]:
linear_svr = make_pipeline(
    StandardScaler(),
    LinearSVR(C=10, epsilon=0.1, max_iter=10000, random_state=42)
)
linear_svr.fit(X_train, y_train_scaled)
linear_svr_test_pred_scaled = linear_svr.predict(X_test)

print("LinearSVR test R²:", r2_score(y_test_scaled, linear_svr_test_pred_scaled))

## 7. Regression metrics in the original target units

The predictions below come from the selected GridSearchCV model and are converted back to the original Diabetes target scale.

In [ ]:
best_y_test_pred = y_scaler.inverse_transform(
    best_y_test_pred_scaled.reshape(-1, 1)
).ravel()
y_test_original = y_test.to_numpy()

print("MAE:", mean_absolute_error(y_test_original, best_y_test_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test_original, best_y_test_pred)))
print("R²:", r2_score(y_test_scaled, best_y_test_pred_scaled))

## Key takeaways

- Standardize input features for SVM regression and fit scalers using training data only.
- Use GridSearchCV on training data for hyperparameter selection.
- Use `grid_search.best_estimator_` to evaluate the model selected by cross-validation.
- MAE and RMSE are reported in the original target units; R² is scale invariant.
